# 04 - Unity Catalog governance for PHI

| Control | Where | Purpose |
| --- | --- | --- |
| **Tags** `phi`, `phi_type` | PHI columns in the bronze, silver and worklist tables | Make PHI discoverable and auditable; drive policies |
| **Comments** | Every table (set in pipeline code and notebooks) | Used by Genie and by people browsing the catalog |
| **Column masks** | `coordinator_worklist`: name, MRN, DOB | Analysts see masked identifiers; only care-team members with `can_view_phi` see clear values |
| **Row filter** | `coordinator_worklist.facility_id` | Coordinators see only their own hospital's patients |
| **Entitlement table** | `lakeview_ml.facility_access` | One place to manage who can see which facility and whether they can see PHI |
| **Lineage and audit** | Automatic in Unity Catalog, plus `system.access.audit` | HIPAA access reviews |

In [0]:
%sql
-- Silver patient master (pipeline materialized view)
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_patients ALTER COLUMN mrn        SET TAGS ('phi' = 'true', 'phi_type' = 'identifier');
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_patients ALTER COLUMN first_name SET TAGS ('phi' = 'true', 'phi_type' = 'name');
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_patients ALTER COLUMN last_name  SET TAGS ('phi' = 'true', 'phi_type' = 'name');
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_patients ALTER COLUMN dob        SET TAGS ('phi' = 'true', 'phi_type' = 'date_of_birth');
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_patients ALTER COLUMN zip        SET TAGS ('phi' = 'true', 'phi_type' = 'geography');
ALTER MATERIALIZED VIEW simo_show_case_catalog.lakeview_health.silver_discharge_notes ALTER COLUMN note_text SET TAGS ('phi' = 'true', 'phi_type' = 'free_text');
-- Bronze raw tables (pipeline streaming tables)
ALTER STREAMING TABLE simo_show_case_catalog.lakeview_health.bronze_patients ALTER COLUMN mrn SET TAGS ('phi' = 'true', 'phi_type' = 'identifier');
ALTER STREAMING TABLE simo_show_case_catalog.lakeview_health.bronze_patients ALTER COLUMN dob SET TAGS ('phi' = 'true', 'phi_type' = 'date_of_birth');
ALTER STREAMING TABLE simo_show_case_catalog.lakeview_health.bronze_discharge_notes ALTER COLUMN note_text SET TAGS ('phi' = 'true', 'phi_type' = 'free_text');
-- Worklist (served to coordinators)
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN mrn        SET TAGS ('phi' = 'true', 'phi_type' = 'identifier');
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN first_name SET TAGS ('phi' = 'true', 'phi_type' = 'name');
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN last_name  SET TAGS ('phi' = 'true', 'phi_type' = 'name');
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN dob        SET TAGS ('phi' = 'true', 'phi_type' = 'date_of_birth');
ALTER SCHEMA simo_show_case_catalog.lakeview_health SET TAGS ('contains_phi' = 'true', 'lakeview_domain' = 'clinical');
ALTER SCHEMA simo_show_case_catalog.lakeview_ml SET TAGS ('contains_phi' = 'true', 'lakeview_domain' = 'care_management');

In [0]:
%sql
CREATE TABLE IF NOT EXISTS simo_show_case_catalog.lakeview_ml.facility_access (
  principal STRING NOT NULL COMMENT 'User email or service principal application id',
  facility_id STRING NOT NULL COMMENT 'LVH-NORTH | LVH-CENTRAL | LVH-SOUTH, or * for all facilities',
  can_view_phi BOOLEAN NOT NULL COMMENT 'true = sees clear name/MRN/DOB in coordinator_worklist'
) COMMENT 'Care-team entitlements: which facility worklists a person may see and whether they can see PHI. Drives the row filter and column masks on coordinator_worklist.';

-- Row filter: only rows for facilities the caller is entitled to
CREATE OR REPLACE FUNCTION simo_show_case_catalog.lakeview_ml.rf_facility(facility STRING)
RETURNS BOOLEAN
COMMENT 'Row filter for coordinator_worklist: caller must be entitled to the facility in lakeview_ml.facility_access'
RETURN EXISTS (SELECT 1 FROM simo_show_case_catalog.lakeview_ml.facility_access a
               WHERE a.principal = current_user() AND (a.facility_id = facility OR a.facility_id = '*'));

-- Column masks: clear values only for entitled care-team members with can_view_phi
CREATE OR REPLACE FUNCTION simo_show_case_catalog.lakeview_ml.mask_phi_string(v STRING)
RETURNS STRING
COMMENT 'Column mask: returns *** unless caller has can_view_phi in lakeview_ml.facility_access'
RETURN CASE WHEN EXISTS (SELECT 1 FROM simo_show_case_catalog.lakeview_ml.facility_access a WHERE a.principal = current_user() AND a.can_view_phi)
            THEN v ELSE '***' END;

CREATE OR REPLACE FUNCTION simo_show_case_catalog.lakeview_ml.mask_phi_dob(d DATE)
RETURNS DATE
COMMENT 'Column mask: DOB generalized to January 1 of birth year unless caller has can_view_phi'
RETURN CASE WHEN EXISTS (SELECT 1 FROM simo_show_case_catalog.lakeview_ml.facility_access a WHERE a.principal = current_user() AND a.can_view_phi)
            THEN d ELSE make_date(year(d), 1, 1) END;

In [0]:
%sql
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist SET ROW FILTER simo_show_case_catalog.lakeview_ml.rf_facility ON (facility_id);
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN first_name SET MASK simo_show_case_catalog.lakeview_ml.mask_phi_string;
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN last_name  SET MASK simo_show_case_catalog.lakeview_ml.mask_phi_string;
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN mrn        SET MASK simo_show_case_catalog.lakeview_ml.mask_phi_string;
ALTER TABLE simo_show_case_catalog.lakeview_ml.coordinator_worklist ALTER COLUMN dob        SET MASK simo_show_case_catalog.lakeview_ml.mask_phi_dob;

In [0]:
%sql
-- Simulate a LVH-NORTH user without PHI rights
DELETE FROM simo_show_case_catalog.lakeview_ml.facility_access WHERE principal = current_user();
INSERT INTO simo_show_case_catalog.lakeview_ml.facility_access VALUES (current_user(), 'LVH-NORTH', false);
SELECT facility_id, count(*) AS visible_rows, max(first_name) AS sample_first_name, max(mrn) AS sample_mrn, max(dob) AS sample_dob
FROM simo_show_case_catalog.lakeview_ml.coordinator_worklist GROUP BY facility_id

facility_id,visible_rows,sample_first_name,sample_mrn,sample_dob
LVH-NORTH,90,***,***,1998-01-01


In [0]:
%sql
DELETE FROM simo_show_case_catalog.lakeview_ml.facility_access WHERE principal = current_user();
INSERT INTO simo_show_case_catalog.lakeview_ml.facility_access VALUES (current_user(), '*', true);
SELECT facility_id, count(*) AS visible_rows, max(first_name) AS sample_first_name, max(mrn) AS sample_mrn, max(dob) AS sample_dob
FROM simo_show_case_catalog.lakeview_ml.coordinator_worklist GROUP BY facility_id ORDER BY facility_id

facility_id,visible_rows,sample_first_name,sample_mrn,sample_dob
LVH-CENTRAL,96,William,MRN99488964,2001-08-20
LVH-NORTH,90,Thomas,MRN98063059,1998-02-19
LVH-SOUTH,68,William,MRN99811834,2001-10-27


In [0]:
%sql
SELECT schema_name, table_name, column_name, tag_value AS phi_type
FROM simo_show_case_catalog.information_schema.column_tags
WHERE schema_name IN ('lakeview_health', 'lakeview_ml') AND tag_name = 'phi_type'
ORDER BY schema_name, table_name, column_name

schema_name,table_name,column_name,phi_type
lakeview_health,bronze_discharge_notes,note_text,free_text
lakeview_health,bronze_patients,dob,date_of_birth
lakeview_health,bronze_patients,mrn,identifier
lakeview_health,silver_discharge_notes,note_text,free_text
lakeview_health,silver_patients,dob,date_of_birth
lakeview_health,silver_patients,first_name,name
lakeview_health,silver_patients,last_name,name
lakeview_health,silver_patients,mrn,identifier
lakeview_health,silver_patients,zip,geography
lakeview_ml,coordinator_worklist,dob,date_of_birth
